# 02 Build customer features

For every customer at every cutoff date, build 21 features from purchases before that date only: recency, tenure, orders and spend over the last 30 to 365 days, spend trend, buying rhythm (usual gap between orders and how overdue the customer is), basket size, return rate, autumn buying and location. The code is in src/features.py.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.data import PROCESSED_DIR, load_clean
from src.evaluation import measure_columns
from src.features import build_features
from src.labels import build_snapshots, split_cutoffs, valid_cutoffs

In [2]:
data = load_clean()
cutoffs = valid_cutoffs(90)
train_cuts, _ = split_cutoffs(cutoffs, horizon_days=90, n_test=3)

snapshots = build_snapshots(data.purchases, cutoffs, horizon_days=90)
features = build_features(data.purchases, data.cancellations, snapshots)
features.to_parquet(PROCESSED_DIR / "features.parquet", index=False)
features.shape

(42850, 25)

## Churn rate by feature

Training months only. Customers are split into five equal groups by each feature, from lowest (Q1) to highest (Q5).

In [3]:
train = features[features["cutoff"].isin(train_cuts)]

def churn_by_group(col):
    x = train[col].dropna()
    if x.nunique() <= 2:
        return train.groupby(col)["churn"].mean().rename(lambda v: f"={v:g}")
    groups = pd.qcut(x.rank(method="first"), 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
    return train.loc[x.index].groupby(groups, observed=True)["churn"].mean()

pd.DataFrame({c: churn_by_group(c) for c in measure_columns(features)}).T.round(2)

,=0,=1,Q1,Q2,Q3,Q4,Q5
recency_days,NaN,NaN,0.40,0.56,0.67,0.76,0.85
tenure_days,NaN,NaN,0.75,0.76,0.70,0.57,0.47
orders_30d,NaN,NaN,0.75,0.73,0.70,0.68,0.39
spend_30d,NaN,NaN,0.75,0.73,0.70,0.68,0.38
orders_90d,NaN,NaN,0.82,0.76,0.74,0.60,0.31
spend_90d,NaN,NaN,0.82,0.76,0.73,0.60,0.33
orders_180d,NaN,NaN,0.85,0.80,0.73,0.60,0.27
spend_180d,NaN,NaN,0.85,0.80,0.71,0.58,0.30
orders_365d,NaN,NaN,0.87,0.81,0.73,0.57,0.26
spend_365d,NaN,NaN,0.86,0.81,0.72,0.56,0.30


## Findings

Recency and order count separate churners most clearly: churn rises from 40% for the most recent buyers to 85% for the least recent, and falls from 87% to 26% as orders go up. Customers far past their usual gap between orders also churn more.